# Stage 1 — Prepare Analytic Datasets (Drive outputs)

**Project:** Dialysis echocardiography mortality/hospitalization study  
**Purpose:** Prepare clean, documented analytic datasets from the approved Stage 0 source-of-truth dataset.

## What this notebook does
- Uses `stage0_updated_clean_flat.csv` as the **only source of truth**.
- Applies the approved decision log through DEC-043.
- Treats comorbidity missingness as absence for binary covariates.
- Treats unusable echo values as missing/effective missing, not as normal.
- Excludes same-day echo/death and echo-after-death cases from analytic cohorts, while retaining them in logs.
- Applies clinician-approved echo category grouping.
- Exports outcome-specific datasets and QA logs.

## What this notebook does **not** do
- No final Cox/negative-binomial/logistic models.
- No clustering or feature selection based on outcomes.
- No imputation using mode/mean/median for implausible continuous outliers.
- No use of the second LLM file `01_clean_flat.csv` as source data.

**Output location:** `MyDrive/dialysis/outputs/params/stage1`


In [1]:
# ============================================================
# 1. Setup — Google Drive input/output folders
# ============================================================
from pathlib import Path
import json
from datetime import datetime
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# ------------------------------------------------------------
# Colab / Google Drive paths
# ------------------------------------------------------------
# Approved folder structure:
#   Inputs from Stage 0:  MyDrive/dialysis/outputs/params/stage0
#   Outputs for Stage 1: MyDrive/dialysis/outputs/params/stage1
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except Exception:
    # Allows the notebook to run outside Colab for testing.
    DRIVE_ROOT = Path('.')

# Stage 1 input files are Stage 0 outputs, so they live in the Stage 0 folder.
BASE_DIR = DRIVE_ROOT / 'dialysis' / 'outputs' / 'params' / 'stage0'

# All Stage 1 outputs are written to the Stage 1 folder.
OUT_DIR = DRIVE_ROOT / 'dialysis' / 'outputs' / 'params' / 'stage1'
BASE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Required / optional input files
# ------------------------------------------------------------
INPUT_DATA = BASE_DIR / 'stage0_updated_clean_flat.csv'

# Decision log: accept either final filename, preferring the Stage 1-ready name.
DECISION_LOG_CANDIDATES = [
    BASE_DIR / 'stage0_decision_log_FINAL_for_stage1.csv',
    BASE_DIR / 'stage0_decision_log_FINAL_DEC038_DEC043.csv',
]
DECISION_LOG_FILE = next((p for p in DECISION_LOG_CANDIDATES if p.exists()), DECISION_LOG_CANDIDATES[0])

CATEGORY_MAPPING_FILE = BASE_DIR / 'stage1_clinician_category_mapping_DEC037.csv'

# Optional Stage 0 final variable-list files. The notebook will run even if these are absent,
# but the exports are best when they are available.
MAIN_LIST_CANDIDATES = [
    BASE_DIR / 'stage0_stage1_eligible_main_FINAL.csv',
    BASE_DIR / 'stage0_updated_stage1_eligible_main.csv',
    BASE_DIR / 'inventory_main.csv',
]
SENS_LIST_CANDIDATES = [
    BASE_DIR / 'stage0_sensitivity_exploratory_FINAL.csv',
    BASE_DIR / 'stage0_updated_sensitivity_exploratory.csv',
    BASE_DIR / 'inventory_sensitivity.csv',
]
EXCL_LIST_CANDIDATES = [
    BASE_DIR / 'stage0_descriptive_excluded_FINAL.csv',
    BASE_DIR / 'stage0_updated_descriptive_excluded.csv',
    BASE_DIR / 'inventory_excluded.csv',
]
QA_FILE_CANDIDATES = [
    BASE_DIR / 'stage0_updated_QA_flags.csv',
    BASE_DIR / '04_qa_outliers.csv',
]

REQUIRED_INPUTS = [INPUT_DATA]
missing_required = [str(p) for p in REQUIRED_INPUTS if not p.exists()]
if missing_required:
    raise FileNotFoundError(
        'Missing required Stage 0 input file(s): ' + ', '.join(missing_required) +
        '\nPut stage0_updated_clean_flat.csv in MyDrive/dialysis/outputs/params/stage0, or update BASE_DIR.'
    )

run_metadata = {
    'run_datetime': datetime.now().isoformat(timespec='seconds'),
    'input_data': str(INPUT_DATA),
    'input_dir': str(BASE_DIR),
    'output_dir': str(OUT_DIR),
    'notebook': 'stage1_prepare_analytic_datasets_FIXED_v2_DRIVE_stage0_inputs.ipynb',
    'decision_scope': 'DEC-038 through DEC-044 plus prior Stage 0 decisions',
    'approved_input_path': 'MyDrive/dialysis/outputs/params/stage0',
    'approved_output_path': 'MyDrive/dialysis/outputs/params/stage1',
}

print('Input directory:', BASE_DIR)
print('Input data:', INPUT_DATA)
print('Decision log:', DECISION_LOG_FILE)
print('Category mapping:', CATEGORY_MAPPING_FILE)
print('Output directory:', OUT_DIR)


Mounted at /content/drive
Input directory: /content/drive/MyDrive/dialysis/outputs/params/stage0
Input data: /content/drive/MyDrive/dialysis/outputs/params/stage0/stage0_updated_clean_flat.csv
Decision log: /content/drive/MyDrive/dialysis/outputs/params/stage0/stage0_decision_log_FINAL_for_stage1.csv
Category mapping: /content/drive/MyDrive/dialysis/outputs/params/stage0/stage1_clinician_category_mapping_DEC037.csv
Output directory: /content/drive/MyDrive/dialysis/outputs/params/stage1


In [2]:
# ============================================================
# 2. Load inputs and validate source dataset
# ============================================================
df_raw = pd.read_csv(INPUT_DATA)
df = df_raw.copy()

print('Input shape:', df.shape)
assert 'patient_id' in df.columns, 'patient_id column is required.'
assert df['patient_id'].is_unique, 'patient_id must be unique.'
assert len(df) == 645, f'Expected 645 rows before exclusions; found {len(df)}.'

# Helper: load optional variable list files.
def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

def load_var_list(paths, label):
    p = first_existing(paths)
    if p is None:
        print(f'Optional {label} variable list not found; continuing with empty list.')
        return pd.DataFrame(), []
    tmp = pd.read_csv(p)
    if 'column' in tmp.columns:
        cols = tmp['column'].dropna().astype(str).tolist()
    elif 'variable' in tmp.columns:
        cols = tmp['variable'].dropna().astype(str).tolist()
    else:
        cols = tmp.iloc[:,0].dropna().astype(str).tolist()
    print(f'Loaded {label}: {p.name} ({len(cols)} variables)')
    return tmp, cols

main_list_df, main_vars_stage0 = load_var_list(MAIN_LIST_CANDIDATES, 'main')
sens_list_df, sens_vars_stage0 = load_var_list(SENS_LIST_CANDIDATES, 'sensitivity/exploratory')
excl_list_df, excl_vars_stage0 = load_var_list(EXCL_LIST_CANDIDATES, 'descriptive/excluded')

# Decision log and mapping are strongly recommended but not required for basic validation.
decision_log = pd.read_csv(DECISION_LOG_FILE) if DECISION_LOG_FILE.exists() else pd.DataFrame()
cat_map = pd.read_csv(CATEGORY_MAPPING_FILE) if CATEGORY_MAPPING_FILE.exists() else pd.DataFrame()

validation_rows = []
for item, ok, detail in [
    ('n_rows_645', len(df) == 645, len(df)),
    ('patient_id_present', 'patient_id' in df.columns, 'patient_id' in df.columns),
    ('patient_id_unique', df['patient_id'].is_unique, df['patient_id'].duplicated().sum()),
    ('decision_log_loaded', not decision_log.empty, DECISION_LOG_FILE.name if DECISION_LOG_FILE.exists() else 'missing'),
    ('category_mapping_loaded', not cat_map.empty, CATEGORY_MAPPING_FILE.name if CATEGORY_MAPPING_FILE.exists() else 'missing'),
]:
    validation_rows.append({'check': item, 'passed': bool(ok), 'detail': detail})

validation = pd.DataFrame(validation_rows)
validation.to_csv(OUT_DIR / 'stage1_input_validation.csv', index=False)
validation

Input shape: (645, 91)
Loaded main: stage0_stage1_eligible_main_FINAL.csv (27 variables)
Loaded sensitivity/exploratory: stage0_sensitivity_exploratory_FINAL.csv (30 variables)
Loaded descriptive/excluded: stage0_descriptive_excluded_FINAL.csv (45 variables)


,check,passed,detail
0,n_rows_645,True,645
1,patient_id_present,True,True
2,patient_id_unique,True,0
3,decision_log_loaded,True,stage0_decision_log_FINAL_for_stage1.csv
4,category_mapping_loaded,True,stage1_clinician_category_mapping_DEC037.csv


In [3]:
# ============================================================
# 3. Standard helpers: dates, effective missingness, numeric coercion
# ============================================================
ECHO_MISSING_TOKENS = {
    '', ' ', 'nan', 'NaN', 'None', 'NO VALUE', 'No Value', 'no value',
    'SEE BELOW', 'See below', 'see below', 'N/A', 'NA', 'Not available',
    'Not Available', 'not available', 'Not well seen', 'not well seen'
}

def clean_string_cell(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        y = x.strip()
        return np.nan if y in ECHO_MISSING_TOKENS else y
    return x

def canonical_echo_category(x):
    """Normalize observed echo category labels before clinician mapping.

    This handles harmless spelling/format variants so clinician decisions apply to
    the exact dataset labels, e.g. Mild-to-moderate (I-II) and
    Moderately-severe (III). It does not collapse categories beyond the
    clinician-approved mapping table.
    """
    x = clean_string_cell(x)
    if pd.isna(x):
        return np.nan
    y = str(x).strip()
    aliases = {
        'Mild-to-moderate (I-II)': 'Mild-to-moderate',
        'Mild-to-mod (I-II)': 'Mild-to-mod',
        'Mild-to-moderate': 'Mild-to-moderate',
        'Mild-to-mod': 'Mild-to-mod',
        'Moderately-severe (III)': 'Mod-severe (III)',
        'Moderate-severe (III)': 'Mod-severe (III)',
        'Mod-severe (III)': 'Mod-severe (III)',
    }
    return aliases.get(y, y)

def effective_missing_mask(series):
    return series.isna() | series.astype(str).str.strip().isin(ECHO_MISSING_TOKENS)

def to_numeric(series):
    return pd.to_numeric(series, errors='coerce')

def parse_date(series):
    return pd.to_datetime(series, errors='coerce')

# Parse key dates.
for col in ['Dialysis_Start_Date', 'Echo_Date', 'DeathDate', 'data_cutoff_date', 'censor_or_event_date']:
    if col in df.columns:
        df[col + '_dt'] = parse_date(df[col])

# Numeric fields frequently needed downstream.
for col in ['time_to_event_days', 'followup_days', 'days_echo_to_dialysis', 'hosp_total', 'hospitalization-count', 'died_1year', 'event', 'event_1y']:
    if col in df.columns:
        df[col] = to_numeric(df[col])

In [4]:
# ============================================================
# 4. DEC-038: Comorbidity missingness means disease absence for binary covariates
# ============================================================
COMORBIDITY_DATE_COLS = [
    'MI', 'CABG', 'IHD', 'AFIB', 'HTN', 'Diabetes mellitus',
    'DYSLIPIDEMIA', 'COPD', 'OncologicalDiagnosis'
]

comorbidity_check_rows = []
for base in COMORBIDITY_DATE_COLS:
    bin_col = f'{base}_binary'
    if base in df.columns:
        expected = (~df[base].isna()).astype(int)
        if bin_col not in df.columns:
            df[bin_col] = expected
            action = 'created_from_nonmissing_source'
        else:
            # Keep existing variable but verify it is consistent enough to audit.
            existing = to_numeric(df[bin_col]).fillna(0).astype(int)
            mismatch = int((existing != expected).sum())
            action = 'verified_existing_binary'
            df[bin_col] = existing
        comorbidity_check_rows.append({
            'source_column': base,
            'binary_column': bin_col,
            'source_present': True,
            'binary_present_after_processing': bin_col in df.columns,
            'n_positive_binary': int(df[bin_col].sum()) if bin_col in df.columns else np.nan,
            'n_missing_source_interpreted_as_absence': int(df[base].isna().sum()),
            'action': action,
            'note': 'DEC-038: missing comorbidity documentation interpreted as absence for binary covariates.'
        })
    elif bin_col in df.columns:
        df[bin_col] = to_numeric(df[bin_col]).fillna(0).astype(int)
        comorbidity_check_rows.append({
            'source_column': base,
            'binary_column': bin_col,
            'source_present': False,
            'binary_present_after_processing': True,
            'n_positive_binary': int(df[bin_col].sum()),
            'n_missing_source_interpreted_as_absence': np.nan,
            'action': 'binary_available_no_source_column',
            'note': 'Binary variable retained; source column not available.'
        })

comorbidity_check = pd.DataFrame(comorbidity_check_rows)
comorbidity_check.to_csv(OUT_DIR / 'stage1_comorbidity_binary_check.csv', index=False)
comorbidity_check

,source_column,binary_column,source_present,binary_present_after_processing,n_positive_binary,n_missing_source_interpreted_as_absence,action,note
0,MI,MI_binary,True,True,193,452,verified_existing_binary,DEC-038: missing comorbidity documentation int...
1,CABG,CABG_binary,True,True,133,512,verified_existing_binary,DEC-038: missing comorbidity documentation int...
2,IHD,IHD_binary,True,True,396,249,verified_existing_binary,DEC-038: missing comorbidity documentation int...
3,AFIB,AFIB_binary,True,True,303,342,verified_existing_binary,DEC-038: missing comorbidity documentation int...
4,HTN,HTN_binary,True,True,533,112,verified_existing_binary,DEC-038: missing comorbidity documentation int...
5,Diabetes mellitus,Diabetes mellitus_binary,True,True,410,235,verified_existing_binary,DEC-038: missing comorbidity documentation int...
6,DYSLIPIDEMIA,DYSLIPIDEMIA_binary,True,True,311,334,verified_existing_binary,DEC-038: missing comorbidity documentation int...
7,COPD,COPD_binary,True,True,105,540,verified_existing_binary,DEC-038: missing comorbidity documentation int...
8,OncologicalDiagnosis,OncologicalDiagnosis_binary,True,True,143,502,verified_existing_binary,DEC-038: missing comorbidity documentation int...


In [5]:
# ============================================================
# 5. DEC-039: Echo effective missingness report
# ============================================================
ECHO_CATEGORICAL_COLS = [
    'LeftVentricleCavitySize', 'LeftVentricleWallThickness', 'LeftVentricleSystolicFunction',
    'RVSize', 'RVSystolicFunction', 'LACavitySize', 'AorticValveStructure',
    'AorticValveRegurgitation', 'MitralValveStructure', 'MitralRegurgitation',
    'TricuspidValveStructure', 'TricuspidRegurgitation', 'ECHO_SPAP'
]
ECHO_NUMERIC_COLS = [
    'LV_EF', 'LeftVentricleEndDiastolicDiameter', 'LeftVentricleEndSystolicDiameter',
    'LeftVentricleInterventricularSeptumThickness', 'LeftVentriclePosteriorWallThickness',
    'LeftVentricleEstimatedMass', 'LeftVentricleEstimatedMassIndex', 'LeftVentricleScoreIndex',
    'EstimatedSysPAPressure', 'MitralInflowPeakEWave',
    'TissueDopplerSVelocitySeptal', 'TissueDopplerEVelositySeptal', 'TissueDopplerEERatioSeptal',
    'TissueDopplerSVelocityLateral', 'TissueDopplerEVelosityLateral', 'TissueDopplerEERatioLateral'
]

# Clean only analysis-ready copies of categorical echo columns; original columns are retained in df_raw.
for col in ECHO_CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].apply(clean_string_cell)

for col in ECHO_NUMERIC_COLS:
    if col in df.columns:
        df[col] = to_numeric(df[col])

echo_missing_rows = []
for col in [c for c in ECHO_CATEGORICAL_COLS + ECHO_NUMERIC_COLS if c in df.columns]:
    eff_mask_raw = effective_missing_mask(df_raw[col]) if col in df_raw.columns else df[col].isna()
    echo_missing_rows.append({
        'variable': col,
        'n': len(df),
        'n_missing_standard_after_processing': int(df[col].isna().sum()),
        'pct_missing_standard_after_processing': round(float(df[col].isna().mean()*100), 2),
        'n_effective_missing_from_raw': int(eff_mask_raw.sum()),
        'pct_effective_missing_from_raw': round(float(eff_mask_raw.mean()*100), 2),
        'decision': 'DEC-039: unusable echo values treated as missing, not normal.'
    })

echo_eff_missing = pd.DataFrame(echo_missing_rows)
echo_eff_missing.to_csv(OUT_DIR / 'stage1_echo_effective_missingness.csv', index=False)
echo_eff_missing.head(20)

,variable,n,n_missing_standard_after_processing,pct_missing_standard_after_processing,n_effective_missing_from_raw,pct_effective_missing_from_raw,decision
0,LeftVentricleCavitySize,645,74,11.47,74,11.47,DEC-039: unusable echo values treated as missi...
1,LeftVentricleWallThickness,645,243,37.67,243,37.67,DEC-039: unusable echo values treated as missi...
2,LeftVentricleSystolicFunction,645,75,11.63,75,11.63,DEC-039: unusable echo values treated as missi...
3,RVSize,645,345,53.49,345,53.49,DEC-039: unusable echo values treated as missi...
4,RVSystolicFunction,645,341,52.87,341,52.87,DEC-039: unusable echo values treated as missi...
5,LACavitySize,645,43,6.67,43,6.67,DEC-039: unusable echo values treated as missi...
6,AorticValveStructure,645,195,30.23,195,30.23,DEC-039: unusable echo values treated as missi...
7,AorticValveRegurgitation,645,343,53.18,343,53.18,DEC-039: unusable echo values treated as missi...
8,MitralValveStructure,645,463,71.78,463,71.78,DEC-039: unusable echo values treated as missi...
9,MitralRegurgitation,645,48,7.44,48,7.44,DEC-039: unusable echo values treated as missi...


In [6]:
# ============================================================
# 6. DEC-040: Exclusion flags for same-day echo/death or echo after death
# ============================================================
if {'Echo_Date_dt', 'DeathDate_dt'}.issubset(df.columns):
    echo_date = df['Echo_Date_dt']
    death_date = df['DeathDate_dt']
    df['flag_echo_after_death_stage1'] = ((~echo_date.isna()) & (~death_date.isna()) & (echo_date.dt.date > death_date.dt.date)).astype(int)
    df['flag_echo_same_day_as_death_stage1'] = ((~echo_date.isna()) & (~death_date.isna()) & (echo_date.dt.date == death_date.dt.date)).astype(int)
else:
    df['flag_echo_after_death_stage1'] = 0
    df['flag_echo_same_day_as_death_stage1'] = 0

df['exclude_same_day_or_after_death_stage1'] = ((df['flag_echo_after_death_stage1'] == 1) | (df['flag_echo_same_day_as_death_stage1'] == 1)).astype(int)

exclusion_rows = []
for _, row in df.loc[df['exclude_same_day_or_after_death_stage1'] == 1].iterrows():
    reason = 'echo_after_death' if row['flag_echo_after_death_stage1'] == 1 else 'echo_same_calendar_date_as_death'
    exclusion_rows.append({
        'patient_id': row['patient_id'],
        'exclusion_reason': reason,
        'stage': 'Stage 1',
        'applies_to_analysis': 'main, one-year mortality, survival, hospitalization',
        'excluded_from_main': True,
        'excluded_from_oneyear_mortality': True,
        'excluded_from_survival': True,
        'excluded_from_hospitalization': True,
        'Echo_Date': row.get('Echo_Date'),
        'DeathDate': row.get('DeathDate'),
        'decision': 'DEC-040'
    })

exclusions_log = pd.DataFrame(exclusion_rows)
if exclusions_log.empty:
    exclusions_log = pd.DataFrame(columns=['patient_id','exclusion_reason','stage','applies_to_analysis','excluded_from_main','excluded_from_oneyear_mortality','excluded_from_survival','excluded_from_hospitalization','Echo_Date','DeathDate','decision'])
exclusions_log.to_csv(OUT_DIR / 'stage1_exclusions_log.csv', index=False)

exclusions_summary = pd.DataFrame({
    'metric': ['n_echo_after_death', 'n_echo_same_day_as_death', 'n_excluded_same_day_or_after_death'],
    'n': [
        int(df['flag_echo_after_death_stage1'].sum()),
        int(df['flag_echo_same_day_as_death_stage1'].sum()),
        int(df['exclude_same_day_or_after_death_stage1'].sum())
    ],
})
exclusions_summary['pct_of_645'] = round(exclusions_summary['n'] / len(df) * 100, 2)
exclusions_summary.to_csv(OUT_DIR / 'stage1_exclusions_summary.csv', index=False)
exclusions_summary

,metric,n,pct_of_645
0,n_echo_after_death,0,0.00
1,n_echo_same_day_as_death,1,0.16
2,n_excluded_same_day_or_after_death,1,0.16


In [7]:
# ============================================================
# 7. DEC-041: Echo-to-dialysis timing distribution
# ============================================================
if 'days_echo_to_dialysis' not in df.columns and 'gap_echo_to_dial_days' in df.columns:
    df['days_echo_to_dialysis'] = to_numeric(df['gap_echo_to_dial_days']) * -1  # only if needed; verify externally

if 'days_echo_to_dialysis' in df.columns:
    df['days_echo_to_dialysis'] = to_numeric(df['days_echo_to_dialysis'])
    d = df['days_echo_to_dialysis']
    timing_summary = pd.DataFrame([{
        'n_nonmissing': int(d.notna().sum()),
        'mean_days': d.mean(),
        'median_days': d.median(),
        'sd_days': d.std(),
        'min_days': d.min(),
        'q25_days': d.quantile(0.25),
        'q75_days': d.quantile(0.75),
        'max_days': d.max(),
        'n_after_dialysis': int((d > 0).sum()),
        'n_same_day': int((d == 0).sum()),
        'n_before_dialysis': int((d < 0).sum()),
        'decision': 'DEC-041: no maximum gap cutoff at Stage 1; summarize and present distribution.'
    }])

    bins = [-np.inf, -365, -180, -90, -30, -1, 0, np.inf]
    labels = ['before_gt_365d', 'before_181_365d', 'before_91_180d', 'before_31_90d', 'before_0_30d', 'same_day', 'after_dialysis']
    df['echo_to_dialysis_timing_category'] = pd.cut(d, bins=bins, labels=labels, include_lowest=True, right=True)
    # pd.cut assigns 0 to same_day because bin [-1,0]; relabel exact zero to same_day, positive to after.
    df.loc[d == 0, 'echo_to_dialysis_timing_category'] = 'same_day'
    df.loc[d > 0, 'echo_to_dialysis_timing_category'] = 'after_dialysis'

    timing_categories = (df['echo_to_dialysis_timing_category']
                         .value_counts(dropna=False)
                         .rename_axis('timing_category')
                         .reset_index(name='n'))
    timing_categories['pct'] = round(timing_categories['n'] / len(df) * 100, 2)

    extreme_gaps = df.loc[d.abs() > 365, ['patient_id','Echo_Date','Dialysis_Start_Date','days_echo_to_dialysis','echo_to_dialysis_timing_category']].copy()
else:
    timing_summary = pd.DataFrame()
    timing_categories = pd.DataFrame()
    extreme_gaps = pd.DataFrame()

timing_summary.to_csv(OUT_DIR / 'stage1_echo_to_dialysis_timing_summary.csv', index=False)
timing_categories.to_csv(OUT_DIR / 'stage1_echo_to_dialysis_timing_categories.csv', index=False)
extreme_gaps.to_csv(OUT_DIR / 'stage1_echo_to_dialysis_extreme_gaps.csv', index=False)

timing_summary

,n_nonmissing,mean_days,median_days,sd_days,min_days,q25_days,q75_days,max_days,n_after_dialysis,n_same_day,n_before_dialysis,decision
0,645,-117.260465,-70.0,125.089721,-366,-227.0,-5.0,67,107,11,527,DEC-041: no maximum gap cutoff at Stage 1; sum...


In [8]:
# ============================================================
# 8. DEC-030: Continuous outlier handling for Stage 1 datasets
# ============================================================
# Conservative clinical plausibility thresholds. These are action rules for analytic copies, not deletion rules.
OUTLIER_RULES = {
    'BMI': {'low': 12, 'high': 70},
    'Weight': {'low': 30, 'high': 250},
    'LeftVentricleInterventricularSeptumThickness': {'low': 0.4, 'high': 2.5},
    'LeftVentriclePosteriorWallThickness': {'low': 0.4, 'high': 2.0},
    'LeftVentricleEstimatedMass': {'low': 30, 'high': 600},
    'LeftVentricleEstimatedMassIndex': {'low': 20, 'high': 300},
}

outlier_action_rows = []
for col, rule in OUTLIER_RULES.items():
    if col not in df.columns:
        continue
    df[col] = to_numeric(df[col])
    low, high = rule.get('low'), rule.get('high')
    mask = pd.Series(False, index=df.index)
    if low is not None:
        mask |= df[col] < low
    if high is not None:
        mask |= df[col] > high
    for idx, row in df.loc[mask, ['patient_id', col]].iterrows():
        outlier_action_rows.append({
            'patient_id': row['patient_id'],
            'variable': col,
            'original_value': row[col],
            'action': 'set_to_missing_in_stage1_analytic_copy',
            'corrected_value': np.nan,
            'reason': f'Implausible continuous value outside [{low}, {high}] and no approved unit-conversion rule.',
            'requires_clinical_review': True,
            'decision': 'DEC-030'
        })
    df.loc[mask, col] = np.nan

outlier_action_log = pd.DataFrame(outlier_action_rows)
if outlier_action_log.empty:
    outlier_action_log = pd.DataFrame(columns=['patient_id','variable','original_value','action','corrected_value','reason','requires_clinical_review','decision'])
outlier_action_log.to_csv(OUT_DIR / 'stage1_outlier_action_log.csv', index=False)
outlier_action_log.head(30)

,patient_id,variable,original_value,action,corrected_value,reason,requires_clinical_review,decision
0,362.0,BMI,81.216160,set_to_missing_in_stage1_analytic_copy,NaN,"Implausible continuous value outside [12, 70] ...",True,DEC-030
1,407.0,BMI,3.746098,set_to_missing_in_stage1_analytic_copy,NaN,"Implausible continuous value outside [12, 70] ...",True,DEC-030


In [9]:
# ============================================================
# 9. DEC-037 / DEC-043: Clinician-approved category harmonization
# ============================================================
category_counts_before_after_rows = []
variable_processing_rows = []

if not cat_map.empty:
    required_map_cols = {'original_variable','grouped_variable','original_category','grouped_category'}
    missing_cols = required_map_cols - set(cat_map.columns)
    if missing_cols:
        raise ValueError(f'Category mapping file missing required columns: {missing_cols}')

    for orig_var, sub in cat_map.groupby('original_variable'):
        if orig_var not in df.columns:
            variable_processing_rows.append({
                'variable': orig_var,
                'new_variable': None,
                'action': 'mapping_not_applied_variable_missing',
                'decision': 'DEC-037/DEC-043',
                'note': 'Original variable not found in dataset.'
            })
            continue

        grouped_var = sub['grouped_variable'].iloc[0]
        is_no_change = (
            grouped_var == orig_var
            or sub['original_category'].astype(str).str.upper().eq('ALL_CURRENT_CATEGORIES').any()
            or sub['grouped_category'].astype(str).str.lower().eq('no change').any()
        )

        before_counts = df[orig_var].value_counts(dropna=False).reset_index()
        before_counts.columns = ['category', 'n_before']
        before_counts['variable'] = orig_var

        source = df[orig_var].apply(clean_string_cell)

        if is_no_change:
            # Clinicians explicitly requested no grouping. Preserve original categories and
            # log this as intentional, not as an unmapped-category problem.
            df[grouped_var] = source
            unmapped = []
            action = 'no_grouping_requested_original_categories_retained'
            note = 'Clinician decision: no category grouping requested; original cleaned categories retained.'
        else:
            # Build a canonicalized mapping so exact observed labels such as
            # 'Mild-to-moderate (I-II)' and 'Moderately-severe (III)' map correctly.
            mapping = {}
            for _, row in sub.iterrows():
                key = canonical_echo_category(row['original_category'])
                if pd.isna(key):
                    continue
                val = str(row['grouped_category'])
                mapping[key] = val
                # Clinician table sometimes says Mild-to-mod while raw echo labels say
                # Mild-to-moderate (I-II). Treat these as wording variants, not as
                # distinct clinical levels.
                if key == 'Mild-to-mod':
                    mapping['Mild-to-moderate'] = val
                if key == 'Mild-to-moderate':
                    mapping['Mild-to-mod'] = val

            canonical_source = source.apply(canonical_echo_category)
            df[grouped_var] = canonical_source.map(lambda x: np.nan if pd.isna(x) else mapping.get(str(x), str(x)))

            unmapped = sorted(set(canonical_source.dropna().astype(str).unique()) - set(mapping.keys()))
            action = 'clinician_category_mapping_applied'
            note = f'Unmapped categories retained as original labels: {unmapped}' if unmapped else 'All observed mapped categories were covered after canonical label normalization.'

        after_counts = df[grouped_var].value_counts(dropna=False).reset_index()
        after_counts.columns = ['category', 'n_after']
        after_counts['variable'] = grouped_var

        # Long-form before/after counts.
        for _, r in before_counts.iterrows():
            category_counts_before_after_rows.append({
                'original_variable': orig_var,
                'grouped_variable': grouped_var,
                'stage': 'before',
                'category': r['category'],
                'n': int(r['n_before'])
            })
        for _, r in after_counts.iterrows():
            category_counts_before_after_rows.append({
                'original_variable': orig_var,
                'grouped_variable': grouped_var,
                'stage': 'after',
                'category': r['category'],
                'n': int(r['n_after'])
            })

        variable_processing_rows.append({
            'variable': orig_var,
            'new_variable': grouped_var,
            'action': action,
            'decision': 'DEC-037/DEC-043',
            'note': note
        })

cat_counts = pd.DataFrame(category_counts_before_after_rows)
processing_log = pd.DataFrame(variable_processing_rows)

cat_map.to_csv(OUT_DIR / 'stage1_clinician_category_mapping_applied.csv', index=False)
cat_counts.to_csv(OUT_DIR / 'stage1_categorical_counts_before_after.csv', index=False)
processing_log.to_csv(OUT_DIR / 'stage1_variable_processing_log.csv', index=False)

processing_log.head(20)

,variable,new_variable,action,decision,note
0,AorticValveRegurgitation,aortic_regurgitation_clin_grouped,clinician_category_mapping_applied,DEC-037/DEC-043,All observed mapped categories were covered af...
1,AorticValveStructure,aortic_valve_structure_clin_grouped,clinician_category_mapping_applied,DEC-037/DEC-043,All observed mapped categories were covered af...
2,ECHO_SPAP,ECHO_SPAP,no_grouping_requested_original_categories_reta...,DEC-037/DEC-043,Clinician decision: no category grouping reque...
3,LACavitySize,LACavitySize,no_grouping_requested_original_categories_reta...,DEC-037/DEC-043,Clinician decision: no category grouping reque...
4,LeftVentricleCavitySize,lv_cavity_size_clin_grouped,clinician_category_mapping_applied,DEC-037/DEC-043,All observed mapped categories were covered af...
5,LeftVentricleSystolicFunction,LeftVentricleSystolicFunction,no_grouping_requested_original_categories_reta...,DEC-037/DEC-043,Clinician decision: no category grouping reque...
6,LeftVentricleWallThickness,lv_wall_thickness_clin_grouped,clinician_category_mapping_applied,DEC-037/DEC-043,All observed mapped categories were covered af...
7,MitralRegurgitation,mitral_regurgitation_clin_grouped,clinician_category_mapping_applied,DEC-037/DEC-043,All observed mapped categories were covered af...
8,RVSize,rv_size_clin_grouped,clinician_category_mapping_applied,DEC-037/DEC-043,All observed mapped categories were covered af...
9,RVSystolicFunction,RVSystolicFunction,no_grouping_requested_original_categories_reta...,DEC-037/DEC-043,Clinician decision: no category grouping reque...


In [10]:
# ============================================================
# 10. DEC-042: Retain free-text echo summaries, but do not model them now
# ============================================================
TEXT_COLS = [
    'LeftVentricleSummary', 'AorticValveSummary', 'MitralValveSummary', 'ProcedureSummary'
]
text_retention = []
for col in TEXT_COLS:
    if col in df.columns:
        text_retention.append({
            'variable': col,
            'n_nonmissing': int(df[col].notna().sum()),
            'n_unique_nonmissing': int(df[col].nunique(dropna=True)),
            'stage1_role': 'retained_for_future_text_or_NLP_review_not_modeled',
            'decision': 'DEC-042'
        })
text_retention = pd.DataFrame(text_retention)
text_retention.to_csv(OUT_DIR / 'stage1_text_fields_retention_log.csv', index=False)
text_retention

,variable,n_nonmissing,n_unique_nonmissing,stage1_role,decision
0,LeftVentricleSummary,195,52,retained_for_future_text_or_NLP_review_not_mod...,DEC-042
1,AorticValveSummary,138,42,retained_for_future_text_or_NLP_review_not_mod...,DEC-042
2,MitralValveSummary,202,57,retained_for_future_text_or_NLP_review_not_mod...,DEC-042
3,ProcedureSummary,645,644,retained_for_future_text_or_NLP_review_not_mod...,DEC-042


In [11]:
# ============================================================
# 11. Build predictor lists for Stage 1 exports
# ============================================================
OUTCOME_ADMIN_FOLLOWUP = {
    'DeathDate','Dialysis_Start_Date','Echo_Date','data_cutoff_date','censor_or_event_date',
    'event','event_1y','death_event','died_1year','time_to_event_days','time_to_event_years',
    'followup_days','followup_years','log_followup_years','hosp_total','hospitalization-count',
    'flag_echo_after_death','flag_echo_after_death_stage1','flag_echo_same_day_as_death_stage1',
    'exclude_same_day_or_after_death_stage1'
}
DATE_DT_COLS = {c for c in df.columns if c.endswith('_dt')}
TEXT_COL_SET = set(TEXT_COLS)
TIMING_DUPLICATES = {'months_echo_to_dialysis', 'months_dialysis_to_echo', 'gap_echo_to_dial_days'}
ALLOWED_TIMING = {'days_echo_to_dialysis', 'echo_to_dialysis_timing_category'}

# Original -> grouped variable map.
orig_to_grouped = {}
if not cat_map.empty:
    for orig_var, sub in cat_map.groupby('original_variable'):
        if orig_var in df.columns:
            orig_to_grouped[orig_var] = sub['grouped_variable'].iloc[0]

def convert_to_stage1_predictors(vars_in, label):
    out = []
    exclusions = []
    for v in vars_in:
        if v not in df.columns and v not in orig_to_grouped:
            exclusions.append({'source_list': label, 'variable': v, 'reason': 'not_found_in_dataset'})
            continue
        # Replace original categorical variable with clinician-grouped version when available.
        v2 = orig_to_grouped.get(v, v)
        if v2 in OUTCOME_ADMIN_FOLLOWUP or v2 in DATE_DT_COLS or v2 in TEXT_COL_SET or v2 in TIMING_DUPLICATES:
            exclusions.append({'source_list': label, 'variable': v, 'stage1_variable': v2, 'reason': 'not_predictor_or_duplicate'})
            continue
        if v2 not in df.columns:
            exclusions.append({'source_list': label, 'variable': v, 'stage1_variable': v2, 'reason': 'mapped_variable_missing'})
            continue
        if v2 not in out:
            out.append(v2)
    return out, pd.DataFrame(exclusions)

# Fallback main list if Stage 0 list is missing.
if not main_vars_stage0:
    # Minimal conservative fallback: core clinical + selected echo variables.
    main_vars_stage0 = [
        'm/f','AgeAtFirstHFDate','creatinine-numeric result','albumin-numeric result','hb-numeric result',
        'HD/PD','MI_binary','CABG_binary','IHD_binary','AFIB_binary','HTN_binary','Diabetes mellitus_binary','COPD_binary',
        'days_echo_to_dialysis','LV_EF','LeftVentricleSystolicFunction','LACavitySize','TricuspidRegurgitation',
        'MitralRegurgitation','EstimatedSysPAPressure','LeftVentricleEstimatedMassIndex','MitralInflowPeakEWave','TissueDopplerEERatioSeptal'
    ]

main_predictors, main_excl = convert_to_stage1_predictors(main_vars_stage0, 'main')
sens_predictors, sens_excl = convert_to_stage1_predictors(sens_vars_stage0, 'sensitivity_exploratory')

# Enforce DEC-034: do not include GFR and creatinine together in main; keep creatinine in main if both are present.
if 'GFR' in main_predictors and 'creatinine-numeric result' in main_predictors:
    main_predictors.remove('GFR')
    if 'GFR' not in sens_predictors and 'GFR' in df.columns:
        sens_predictors.append('GFR')

predictor_exclusions = pd.concat([main_excl, sens_excl], ignore_index=True) if not main_excl.empty or not sens_excl.empty else pd.DataFrame(columns=['source_list','variable','stage1_variable','reason'])
predictor_exclusions.to_csv(OUT_DIR / 'stage1_predictor_list_exclusions.csv', index=False)

pd.DataFrame({'stage1_main_predictor': main_predictors}).to_csv(OUT_DIR / 'stage1_main_predictor_list.csv', index=False)
pd.DataFrame({'stage1_sensitivity_exploratory_predictor': sens_predictors}).to_csv(OUT_DIR / 'stage1_sensitivity_exploratory_predictor_list.csv', index=False)

print('Main predictors:', len(main_predictors))
print(main_predictors)
print('Sensitivity/exploratory predictors:', len(sens_predictors))

Main predictors: 27
['AFIB_binary', 'AgeAtFirstHFDate', 'CABG_binary', 'COPD_binary', 'Diabetes mellitus_binary', 'HD/PD', 'HTN_binary', 'IHD_binary', 'MI_binary', 'albumin-numeric result', 'creatinine-numeric result', 'crp-numeric result', 'hb-numeric result', 'm/f', 'LV_EF', 'LeftVentricleEstimatedMassIndex', 'MitralInflowPeakEWave', 'TissueDopplerEERatioLateral', 'TissueDopplerEERatioSeptal', 'TissueDopplerEVelosityLateral', 'TissueDopplerEVelositySeptal', 'LACavitySize', 'ECHO_SPAP', 'EstimatedSysPAPressure', 'tricuspid_regurgitation_clin_grouped', 'mitral_regurgitation_clin_grouped', 'days_echo_to_dialysis']
Sensitivity/exploratory predictors: 30


In [12]:
# ============================================================
# 12. Create outcome-specific analytic cohorts
# ============================================================
analysis_base = df.loc[df['exclude_same_day_or_after_death_stage1'] == 0].copy()

# One-year mortality cohort: defined died_1year only.
one_year = analysis_base.loc[analysis_base['died_1year'].notna()].copy() if 'died_1year' in analysis_base.columns else pd.DataFrame()

# Survival cohort: event and valid nonnegative time.
survival = analysis_base.copy()
if 'event' in survival.columns and 'time_to_event_days' in survival.columns:
    survival = survival.loc[survival['event'].notna() & survival['time_to_event_days'].notna() & (survival['time_to_event_days'] >= 0)].copy()
else:
    survival = pd.DataFrame()

# Hospitalization cohort: hosp_total and positive follow-up.
hosp = analysis_base.copy()
if 'hosp_total' in hosp.columns and 'followup_days' in hosp.columns:
    hosp = hosp.loc[hosp['hosp_total'].notna() & hosp['followup_days'].notna() & (hosp['followup_days'] > 0)].copy()
    hosp['followup_years'] = hosp['followup_days'] / 365.25
    hosp['log_followup_years'] = np.log(hosp['followup_years'])
else:
    hosp = pd.DataFrame()

# Helper to select available columns in desired order.
def select_cols(data, cols):
    seen = []
    for c in cols:
        if c in data.columns and c not in seen:
            seen.append(c)
    return data[seen].copy()

ID_COLS = ['patient_id']
COMMON_FLAGS = ['days_echo_to_dialysis','echo_to_dialysis_timing_category','exclude_same_day_or_after_death_stage1']
ONE_YEAR_OUTCOME = ['died_1year']
SURVIVAL_OUTCOME = ['event','time_to_event_days']
HOSP_OUTCOME = ['hosp_total','followup_days','followup_years','log_followup_years']

stage1_main = select_cols(analysis_base, ID_COLS + main_predictors + COMMON_FLAGS + ONE_YEAR_OUTCOME + SURVIVAL_OUTCOME + HOSP_OUTCOME)
stage1_sensitivity = select_cols(analysis_base, ID_COLS + main_predictors + sens_predictors + COMMON_FLAGS + ONE_YEAR_OUTCOME + SURVIVAL_OUTCOME + HOSP_OUTCOME)
stage1_exploratory = select_cols(analysis_base, ID_COLS + main_predictors + sens_predictors + COMMON_FLAGS + ONE_YEAR_OUTCOME + SURVIVAL_OUTCOME + HOSP_OUTCOME)
stage1_oneyear = select_cols(one_year, ID_COLS + main_predictors + COMMON_FLAGS + ONE_YEAR_OUTCOME)
stage1_survival = select_cols(survival, ID_COLS + main_predictors + COMMON_FLAGS + SURVIVAL_OUTCOME)
stage1_hosp = select_cols(hosp, ID_COLS + main_predictors + COMMON_FLAGS + HOSP_OUTCOME)

stage1_main.to_csv(OUT_DIR / 'stage1_main_analysis.csv', index=False)
stage1_sensitivity.to_csv(OUT_DIR / 'stage1_sensitivity_analysis.csv', index=False)
stage1_exploratory.to_csv(OUT_DIR / 'stage1_exploratory_analysis.csv', index=False)
stage1_oneyear.to_csv(OUT_DIR / 'stage1_oneyear_mortality.csv', index=False)
stage1_survival.to_csv(OUT_DIR / 'stage1_survival_analysis.csv', index=False)
stage1_hosp.to_csv(OUT_DIR / 'stage1_hospitalization_analysis.csv', index=False)

cohort_summary = pd.DataFrame([
    {'dataset': 'input_full', 'n_rows': len(df), 'n_cols': df.shape[1]},
    {'dataset': 'analysis_base_after_DEC040_exclusion', 'n_rows': len(analysis_base), 'n_cols': analysis_base.shape[1]},
    {'dataset': 'stage1_main_analysis', 'n_rows': len(stage1_main), 'n_cols': stage1_main.shape[1]},
    {'dataset': 'stage1_oneyear_mortality', 'n_rows': len(stage1_oneyear), 'n_cols': stage1_oneyear.shape[1]},
    {'dataset': 'stage1_survival_analysis', 'n_rows': len(stage1_survival), 'n_cols': stage1_survival.shape[1]},
    {'dataset': 'stage1_hospitalization_analysis', 'n_rows': len(stage1_hosp), 'n_cols': stage1_hosp.shape[1]},
    {'dataset': 'stage1_sensitivity_analysis', 'n_rows': len(stage1_sensitivity), 'n_cols': stage1_sensitivity.shape[1]},
])
cohort_summary.to_csv(OUT_DIR / 'stage1_cohort_summary.csv', index=False)
cohort_summary

,dataset,n_rows,n_cols
0,input_full,645,107
1,analysis_base_after_DEC040_exclusion,644,107
2,stage1_main_analysis,644,35
3,stage1_oneyear_mortality,617,31
4,stage1_survival_analysis,644,32
5,stage1_hospitalization_analysis,644,34
6,stage1_sensitivity_analysis,644,65


In [13]:
# ============================================================
# 13. Missingness report by cohort and variable role
# ============================================================
def missingness_table(data, dataset_name, variables, role):
    rows = []
    for v in variables:
        if v in data.columns:
            n = len(data)
            nmiss = int(data[v].isna().sum())
            rows.append({
                'dataset': dataset_name,
                'variable': v,
                'role': role,
                'n': n,
                'n_missing': nmiss,
                'pct_missing': round(nmiss / n * 100, 2) if n else np.nan,
                'n_nonmissing': int(data[v].notna().sum()),
                'n_unique_nonmissing': int(data[v].nunique(dropna=True))
            })
    return pd.DataFrame(rows)

miss_tables = []
cohorts = [
    ('full_input_processed', df),
    ('analysis_base', analysis_base),
    ('one_year_mortality', one_year),
    ('survival', survival),
    ('hospitalization', hosp),
]
for name, data in cohorts:
    miss_tables.append(missingness_table(data, name, main_predictors, 'main_predictor'))
    miss_tables.append(missingness_table(data, name, sens_predictors, 'sensitivity_exploratory_predictor'))
    miss_tables.append(missingness_table(data, name, ONE_YEAR_OUTCOME + SURVIVAL_OUTCOME + HOSP_OUTCOME, 'outcome_or_denominator'))

missingness_report = pd.concat([t for t in miss_tables if not t.empty], ignore_index=True)
missingness_report.to_csv(OUT_DIR / 'stage1_missingness_report.csv', index=False)
missingness_report.head(30)

,dataset,variable,role,n,n_missing,pct_missing,n_nonmissing,n_unique_nonmissing
0,full_input_processed,AFIB_binary,main_predictor,645,0,0.00,645,2
1,full_input_processed,AgeAtFirstHFDate,main_predictor,645,0,0.00,645,435
2,full_input_processed,CABG_binary,main_predictor,645,0,0.00,645,2
3,full_input_processed,COPD_binary,main_predictor,645,0,0.00,645,2
4,full_input_processed,Diabetes mellitus_binary,main_predictor,645,0,0.00,645,2
5,full_input_processed,HD/PD,main_predictor,645,0,0.00,645,2
6,full_input_processed,HTN_binary,main_predictor,645,0,0.00,645,2
7,full_input_processed,IHD_binary,main_predictor,645,0,0.00,645,2
8,full_input_processed,MI_binary,main_predictor,645,0,0.00,645,2
9,full_input_processed,albumin-numeric result,main_predictor,645,1,0.16,644,39


In [14]:
# ============================================================
# 14. Summary report and metadata
# ============================================================
run_metadata.update({
    'input_shape_rows': int(df_raw.shape[0]),
    'input_shape_columns': int(df_raw.shape[1]),
    'processed_shape_rows': int(df.shape[0]),
    'processed_shape_columns': int(df.shape[1]),
    'n_excluded_same_day_or_after_death': int(df['exclude_same_day_or_after_death_stage1'].sum()),
    'n_main_predictors': int(len(main_predictors)),
    'n_sensitivity_exploratory_predictors': int(len(sens_predictors)),
    'n_one_year_mortality': int(len(stage1_oneyear)),
    'n_survival': int(len(stage1_survival)),
    'n_hospitalization': int(len(stage1_hosp)),
})

with open(OUT_DIR / 'stage1_run_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(run_metadata, f, indent=2, ensure_ascii=False)

# Compose markdown report.
summary_lines = []
summary_lines.append('# Stage 1 Summary Report')
summary_lines.append('')
summary_lines.append(f'Generated: {run_metadata["run_datetime"]}')
summary_lines.append(f'Input: `{INPUT_DATA.name}`')
summary_lines.append(f'Input rows/columns: {df_raw.shape[0]} / {df_raw.shape[1]}')
summary_lines.append('')
summary_lines.append('## Key decisions applied')
summary_lines.append('- DEC-038: Missing comorbidity documentation interpreted as absence for binary covariates.')
summary_lines.append('- DEC-039: Unusable echo values treated as missing/effective missing, not normal.')
summary_lines.append('- DEC-040: Same-calendar-date echo/death or echo-after-death excluded from analytic cohorts and reported.')
summary_lines.append('- DEC-041: No maximum echo-to-dialysis gap cutoff at Stage 1; distribution exported for clinician review.')
summary_lines.append('- DEC-042: Echo free-text fields retained for future text/NLP review, not modeled now.')
summary_lines.append('- DEC-043: Echo category distributions before/after clinician grouping exported.')
summary_lines.append('')
summary_lines.append('## Cohort sizes')
summary_lines.append(cohort_summary.to_markdown(index=False))
summary_lines.append('')
summary_lines.append('## Exclusion summary')
summary_lines.append(exclusions_summary.to_markdown(index=False))
summary_lines.append('')
summary_lines.append('## Outputs')
for f in sorted(OUT_DIR.glob('*')):
    summary_lines.append(f'- `{f.name}`')

report = '\n'.join(summary_lines)
with open(OUT_DIR / 'stage1_summary_report.md', 'w', encoding='utf-8') as f:
    f.write(report)

print(report[:3000])

# Stage 1 Summary Report

Generated: 2026-05-05T11:48:55
Input: `stage0_updated_clean_flat.csv`
Input rows/columns: 645 / 91

## Key decisions applied
- DEC-038: Missing comorbidity documentation interpreted as absence for binary covariates.
- DEC-039: Unusable echo values treated as missing/effective missing, not normal.
- DEC-040: Same-calendar-date echo/death or echo-after-death excluded from analytic cohorts and reported.
- DEC-041: No maximum echo-to-dialysis gap cutoff at Stage 1; distribution exported for clinician review.
- DEC-042: Echo free-text fields retained for future text/NLP review, not modeled now.
- DEC-043: Echo category distributions before/after clinician grouping exported.

## Cohort sizes
| dataset                              |   n_rows |   n_cols |
|:-------------------------------------|---------:|---------:|
| input_full                           |      645 |      107 |
| analysis_base_after_DEC040_exclusion |      644 |      107 |
| stage1_main_analysis     